# Multilingual Fake News Detector — Colab training

Fine-tune `bert-base-multilingual-cased` on six languages (English, Urdu,
Spanish, German, Chinese, Korean) using Colab's **free GPU**, then download the
exported TensorFlow SavedModel to drop into `./saved_model/` locally.

**Before running:** Runtime ▸ Change runtime type ▸ Hardware accelerator = **GPU (T4)**.

This notebook runs the same `data_prep.py` and `train.py` that ship in the repo —
the only difference from local training is the GPU.

## 1. Get the project files

Either clone your GitHub fork (set `REPO_URL`) **or** comment the clone out and
use the Files panel to upload `data_prep.py`, `train.py`, and the `data/seed/`
CSVs manually.

In [ ]:
REPO_URL = "https://github.com/<your-username>/Fake-News-Detection.git"  # <-- edit me

import os
if REPO_URL and "<your-username>" not in REPO_URL:
    !git clone $REPO_URL project
    %cd project
else:
    print("Set REPO_URL above, or upload data_prep.py, train.py and data/seed/*.csv "
          "via the Files panel, then skip this cell.")

## 2. Install dependencies

Colab already has a recent TensorFlow with GPU support; we only add the
training/data libraries.

In [ ]:
!pip install -q "transformers==4.35.0" "datasets==2.14.6" scikit-learn pandas openpyxl sentencepiece

import tensorflow as tf
print("TF", tf.__version__, "| GPUs:", tf.config.list_physical_devices('GPU'))

## 3. Build the corpus

Downloads the English/Urdu/Spanish datasets from the HuggingFace Hub and merges
them with the committed German/Chinese/Korean seed sets. Drop `--subset` for the
full corpus (recommended on GPU).

In [ ]:
!python data_prep.py            # add e.g. --subset 2000 for a quick run

## 4. Train + export the SavedModel

On a T4 GPU a few epochs over the full corpus take only minutes.

In [ ]:
!python train.py --csv data/train.csv --epochs 3 --batch-size 16

## 5. (Optional) Evaluate

In [ ]:
!python evaluate.py --csv data/test.csv

## 6. Download the SavedModel

Zips `saved_model/` and downloads it. Unzip it into the repo root locally so
that `./saved_model/saved_model.pb` exists, then run `app.py` / `api.py`.

In [ ]:
import shutil
from google.colab import files
shutil.make_archive("saved_model", "zip", "saved_model")
files.download("saved_model.zip")